<a href="https://colab.research.google.com/github/ncrowder/maven/blob/main/maven_drill_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [107]:
import pandas as pd

In [108]:
df = pd.read_csv('https://maven-datasets.s3.us-east-1.amazonaws.com/Data+Drills/OfficeSpace.csv')

In [109]:
df.head()

,Employee Name,Manager Name
0,Bill Lumbergh,NaN
1,Bob Slydell,Bill Lumbergh
2,Bob Porter,Bill Lumbergh
3,Linda M. Grayson,Bill Lumbergh
4,Dom Portwood,Linda M. Grayson


In [110]:
df.columns = ['employee','manager']

In [111]:
df.head()

,employee,manager
0,Bill Lumbergh,NaN
1,Bob Slydell,Bill Lumbergh
2,Bob Porter,Bill Lumbergh
3,Linda M. Grayson,Bill Lumbergh
4,Dom Portwood,Linda M. Grayson


In [112]:
def managers(row):
  bosses = [row['employee']]
  manager = row['manager']
  while pd.notna(manager):
    bosses.append(manager)
    manager = df[df.employee == manager]['manager'].iloc[0]
  return " > ".join(bosses[::-1])

In [113]:
df['Heirarchy'] = df.apply(managers,axis=1)

In [114]:
df

,employee,manager,Heirarchy
0,Bill Lumbergh,NaN,Bill Lumbergh
1,Bob Slydell,Bill Lumbergh,Bill Lumbergh > Bob Slydell
2,Bob Porter,Bill Lumbergh,Bill Lumbergh > Bob Porter
3,Linda M. Grayson,Bill Lumbergh,Bill Lumbergh > Linda M. Grayson
4,Dom Portwood,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Dom Portwood
5,Wendy L. Hargrove,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Wendy L. Ha...
6,Tom Smykowski,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Tom Smykowski
7,Nathan R. Ross,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Nathan R. Ross
8,Peter Gibbons,Dom Portwood,Bill Lumbergh > Linda M. Grayson > Dom Portwoo...
9,Cheryl T. Ackerman,Peter Gibbons,Bill Lumbergh > Linda M. Grayson > Dom Portwoo...


In [75]:
# df['Direct Reports'] = df.groupby('manager')['employee'].transform("count")

In [115]:
direct = df.groupby('manager')['employee'].count().rename_axis('employee').rename('Direct Reports').reset_index()
direct

,employee,Direct Reports
0,Alan B. Peterson,2
1,Bill Lumbergh,3
2,Derek P. Phillips,1
3,Dom Portwood,3
4,Linda M. Grayson,4
5,Nathan C. Carter,3
6,Nathan R. Ross,1
7,Peter Gibbons,1
8,Samir Nagheenanajar,1
9,Tom Smykowski,2


In [116]:
df1 = df.merge(direct, on = 'employee', how = 'left')
df1.fillna(0,inplace = True)
df1

,employee,manager,Heirarchy,Direct Reports
0,Bill Lumbergh,0,Bill Lumbergh,3.0
1,Bob Slydell,Bill Lumbergh,Bill Lumbergh > Bob Slydell,0.0
2,Bob Porter,Bill Lumbergh,Bill Lumbergh > Bob Porter,0.0
3,Linda M. Grayson,Bill Lumbergh,Bill Lumbergh > Linda M. Grayson,4.0
4,Dom Portwood,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Dom Portwood,3.0
5,Wendy L. Hargrove,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Wendy L. Ha...,3.0
6,Tom Smykowski,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Tom Smykowski,2.0
7,Nathan R. Ross,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Nathan R. Ross,1.0
8,Peter Gibbons,Dom Portwood,Bill Lumbergh > Linda M. Grayson > Dom Portwoo...,1.0
9,Cheryl T. Ackerman,Peter Gibbons,Bill Lumbergh > Linda M. Grayson > Dom Portwoo...,0.0


In [117]:
dfexpanded = df.assign(heirarchy = df.Heirarchy.str.split(' > ')).explode('heirarchy')

In [118]:
dfexpanded

,employee,manager,Heirarchy,heirarchy
0,Bill Lumbergh,NaN,Bill Lumbergh,Bill Lumbergh
1,Bob Slydell,Bill Lumbergh,Bill Lumbergh > Bob Slydell,Bill Lumbergh
1,Bob Slydell,Bill Lumbergh,Bill Lumbergh > Bob Slydell,Bob Slydell
2,Bob Porter,Bill Lumbergh,Bill Lumbergh > Bob Porter,Bill Lumbergh
2,Bob Porter,Bill Lumbergh,Bill Lumbergh > Bob Porter,Bob Porter
...,...,...,...,...
23,Bobbie K. Jenkins,Alan B. Peterson,Bill Lumbergh > Linda M. Grayson > Nathan R. R...,Bobbie K. Jenkins
24,Milton Waddams,Tom Smykowski,Bill Lumbergh > Linda M. Grayson > Tom Smykows...,Bill Lumbergh
24,Milton Waddams,Tom Smykowski,Bill Lumbergh > Linda M. Grayson > Tom Smykows...,Linda M. Grayson
24,Milton Waddams,Tom Smykowski,Bill Lumbergh > Linda M. Grayson > Tom Smykows...,Tom Smykowski


In [119]:
total = dfexpanded.groupby('heirarchy').employee.agg(lambda x: x.count() -1).reset_index()
total = total.rename(columns = {'heirarchy':'employee','employee': 'Total Reports'})
total

,employee,Total Reports
0,Alan B. Peterson,2
1,Alice L. Munroe,0
2,Anne Martinez,0
3,Bill Lumbergh,24
4,Bob Porter,0
5,Bob Slydell,0
6,Bobbie K. Jenkins,0
7,Cheryl T. Ackerman,0
8,Derek P. Phillips,1
9,Dom Portwood,8


In [120]:
df2 = df1.merge(total, on = 'employee', how = 'left')
df2

,employee,manager,Heirarchy,Direct Reports,Total Reports
0,Bill Lumbergh,0,Bill Lumbergh,3.0,24
1,Bob Slydell,Bill Lumbergh,Bill Lumbergh > Bob Slydell,0.0,0
2,Bob Porter,Bill Lumbergh,Bill Lumbergh > Bob Porter,0.0,0
3,Linda M. Grayson,Bill Lumbergh,Bill Lumbergh > Linda M. Grayson,4.0,21
4,Dom Portwood,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Dom Portwood,3.0,8
5,Wendy L. Hargrove,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Wendy L. Ha...,3.0,3
6,Tom Smykowski,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Tom Smykowski,2.0,3
7,Nathan R. Ross,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Nathan R. Ross,1.0,3
8,Peter Gibbons,Dom Portwood,Bill Lumbergh > Linda M. Grayson > Dom Portwoo...,1.0,1
9,Cheryl T. Ackerman,Peter Gibbons,Bill Lumbergh > Linda M. Grayson > Dom Portwoo...,0.0,0


In [121]:
df2['Total Reports'].sum()

np.int64(70)